# Quiz Agent Test (Stage 10)

Full pipeline:
1. Load ChromaDB index (retriever)
2. Retrieve relevant chunks for a topic
3. Generate structured quiz questions via `QuizAgent`

Each question includes:
- **question** — review question
- **answer** — correct answer
- **difficulty** — `easy`, `medium`, or `hard`

**Prerequisites:**
- Index already created (`01_test_rag_pipeline.ipynb`, Part A)
- `.env` with `HUGGINGFACE_API_TOKEN` (Stage 7)
- `uv sync`

## Configuration

In [2]:
import json
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VECTOR_DB_PATH = ROOT / "data/vector_db"

TOP_K = 5
QUESTION = "Review key ideas from the Transformer paper"

token = os.getenv("HUGGINGFACE_API_TOKEN") or os.getenv("HF_TOKEN")

print(f"Project root: {ROOT}")
print(f"Vector DB: {VECTOR_DB_PATH}")
print(f"Default topic: {QUESTION}")
print(f"HF token: {'configured' if token else 'NOT configured'}")

Project root: d:\Github\research-workflow-agent
Vector DB: d:\Github\research-workflow-agent\data\vector_db
Default topic: Review key ideas from the Transformer paper
HF token: configured


## 1. Load retriever

In [3]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()

chunk_count = store.collection.count()
print(f"Collection: {store.collection_name} ({chunk_count} chunks)")

if chunk_count == 0:
    raise RuntimeError("Empty index. Run notebook 01_test_rag_pipeline (Part A) first.")

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

d:\Github\research-workflow-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: book_research (52 chunks)


## 2. Retrieve context (RAG)

In [4]:
context = retriever.invoke(QUESTION)

print(f"Topic: {QUESTION}")
print(f"Chunks retrieved: {len(context)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3856.63it/s]


Topic: Review key ideas from the Transformer paper
Chunks retrieved: 5


## 3. Run Quiz Agent

In [5]:
from agents.quiz_agent import QuizAgent

print("Calling Quiz Agent (Hugging Face API)...")
agent = QuizAgent()
result = agent.run(QUESTION, context)

Calling Quiz Agent (Hugging Face API)...


In [6]:
print("Structured JSON output:")
print(result.to_json())

Structured JSON output:
{
  "questions": [
    {
      "question": "What is the primary innovation of the Transformer model?",
      "answer": "Replacing recurrent layers with multi-headed self-attention",
      "difficulty": "easy"
    },
    {
      "question": "What is the name of the paper that presents the Transformer model?",
      "answer": "The Transformer",
      "difficulty": "easy"
    },
    {
      "question": "What is the dimension of the outputs produced by all sub-layers in the Transformer model?",
      "answer": "dmodel = 512",
      "difficulty": "medium"
    },
    {
      "question": "What is the name of the mechanism used in the Transformer model to facilitate residual connections?",
      "answer": "LayerNorm(x + Sublayer(x))",
      "difficulty": "hard"
    },
    {
      "question": "What is the BLEU score achieved by the Transformer model on the English-to-German newstest2014 test?",
      "answer": "27.3",
      "difficulty": "medium"
    },
    {
      "ques

In [7]:
print("=" * 60)
print("QUIZ QUESTIONS")
print("=" * 60)

for i, item in enumerate(result.questions, start=1):
    print(f"\n{i}. [{item.difficulty.upper()}] {item.question}")
    print(f"   Answer: {item.answer}")

QUIZ QUESTIONS

1. [EASY] What is the primary innovation of the Transformer model?
   Answer: Replacing recurrent layers with multi-headed self-attention

2. [EASY] What is the name of the paper that presents the Transformer model?
   Answer: The Transformer

3. [MEDIUM] What is the dimension of the outputs produced by all sub-layers in the Transformer model?
   Answer: dmodel = 512

4. [HARD] What is the name of the mechanism used in the Transformer model to facilitate residual connections?
   Answer: LayerNorm(x + Sublayer(x))

5. [MEDIUM] What is the BLEU score achieved by the Transformer model on the English-to-German newstest2014 test?
   Answer: 27.3

6. [EASY] What is the name of the parser that the Transformer model outperforms even when training only on the WSJ training set of 40K sentences?
   Answer: Berkeley-Parser


## 4. Test another topic

Change `another_topic` and run the cells below.

In [8]:
another_topic = "Self-attention and multi-head attention"

context2 = retriever.invoke(another_topic)
result2 = agent.run(another_topic, context2)

print(json.dumps(result2.to_dict(), indent=2, ensure_ascii=False))

{
  "questions": [
    {
      "question": "What is the primary difference between the attention mechanism used in ConvS2S and ByteNet?",
      "answer": "The distance between positions is linearly calculated for ConvS2S and logarithmically for ByteNet.",
      "difficulty": "medium"
    },
    {
      "question": "What is the purpose of using Multi-Head Attention in the Transformer model?",
      "answer": "To jointly attend to information from different representation subspaces at different positions.",
      "difficulty": "hard"
    },
    {
      "question": "What is the formula for computing the attention function in Scaled Dot-Product Attention?",
      "answer": "softmax(QK^T / √dk) * V",
      "difficulty": "hard"
    },
    {
      "question": "How does the Transformer model use multi-head attention in encoder-decoder attention layers?",
      "answer": "The queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder.",
    